In [1]:
import platform, sys
print(platform.machine(), sys.executable)

arm64 /Users/riteshkc/Desktop/projects/claims_analysis/venv/bin/python


In [2]:
import duckdb
from pathlib import Path

BASE = Path.home() / "Desktop/projects/claims_analysis"
RAW = BASE / "data/raw"
con = duckdb.connect(str(BASE / "data/claims.duckdb"))

In [3]:
con.sql(f"""
CREATE OR REPLACE TABLE members AS
SELECT * EXCLUDE (filename, BENE_BIRTH_DT, BENE_DEATH_DT),
    CAST(regexp_extract(filename, 'DE1_0_([0-9]+)_Ben', 1) AS INT) AS year,
    try_strptime(CAST(BENE_BIRTH_DT AS VARCHAR), '%Y%m%d')::DATE AS birth_dt,
    try_strptime(CAST(BENE_DEATH_DT AS VARCHAR), '%Y%m%d')::DATE AS death_dt
FROM read_csv('{RAW}/DE1_0_20*_Beneficiary_Summary_File_Sample_2.csv', filename=true)
""")

CountBIGINT
343858


In [4]:
def load(table, pattern, dates, amounts):
    fixes = [f"try_strptime({c}, '%Y%m%d')::DATE AS {c}" for c in dates] + \
            [f"TRY_CAST({c} AS DOUBLE) AS {c}" for c in amounts]
    con.execute(f"""
    CREATE OR REPLACE TABLE {table} AS
    SELECT * REPLACE ({", ".join(fixes)})
    FROM read_csv('{RAW}/{pattern}', all_varchar=true)
    """)
    print(table, con.execute(f"SELECT COUNT(*) FROM {table}").fetchall()[0][0])

In [5]:
con = duckdb.connect(str(BASE / "data/claims.duckdb"))

In [6]:
load("inpatient", "*Inpatient_Claims_Sample_2.csv",
     ["CLM_FROM_DT", "CLM_THRU_DT", "CLM_ADMSN_DT", "NCH_BENE_DSCHRG_DT"],
     ["CLM_PMT_AMT", "CLM_UTLZTN_DAY_CNT"])

load("outpatient", "*Outpatient_Claims_Sample_2.csv",
     ["CLM_FROM_DT", "CLM_THRU_DT"], ["CLM_PMT_AMT"])

load("carrier", "*Carrier_Claims_Sample_2*.csv",   # matches both 2A and 2B
     ["CLM_FROM_DT", "CLM_THRU_DT"],
     [f"LINE_NCH_PMT_AMT_{i}" for i in range(1, 14)])

load("rx", "*Prescription_Drug_Events_Sample_2.csv",
     ["SRVC_DT"], ["TOT_RX_CST_AMT", "PTNT_PAY_AMT"])

inpatient 66494
outpatient 792562


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

carrier 4745914
rx 5561154


In [7]:
con.sql("""
SELECT year, COUNT(*) AS members,
       ROUND(AVG(BENE_HI_CVRAGE_TOT_MONS), 1) AS avg_months_covered
FROM members GROUP BY year ORDER BY year
""").df()

,year,members,avg_months_covered
0,2008,116395,11.1
1,2009,114618,11.2
2,2010,112845,11.2


In [8]:
con.sql("""
SELECT MIN(CLM_FROM_DT), MAX(CLM_FROM_DT),
       COUNT(*) FILTER (WHERE CLM_FROM_DT IS NULL) AS bad_dates,
       ROUND(SUM(CLM_PMT_AMT) / 1e6, 1) AS paid_millions
FROM inpatient
""").df()

,min(CLM_FROM_DT),max(CLM_FROM_DT),bad_dates,paid_millions
0,2007-11-27,2010-12-29,60,634.1


In [9]:
con.execute("CALL start_ui()").fetchall()

[('UI started at http://localhost:4213/',)]

In [11]:
con.close()